# 🏥 MedGuard AI — Hackathon Demo Notebook
## Intelligent Medical Records Platform with AI-Augmented Risk Mitigation

**Modules Demonstrated:**
| Module | Inspired By | Purpose |
|--------|-------------|---------|
| 📄 **Data Engine** | Medical Records / FHIR | Synthetic patient record generation |
| 🤖 **JARVIS** | AI-Augmented Engineering | Natural language querying of medical records |
| 🛡️ **GARMA** | GenAI Risk Mitigation Advisor | Drug interactions, risk scoring, anomaly detection |
| 📊 **NotebookLM** | Google NotebookLM | Document ingestion & semantic vector search |

---
### Architecture
```
User Query → 🤖 JARVIS (NL Understanding) 
           → 📊 NotebookLM (Vector Search / RAG)
           → 🛡️ GARMA (Risk Assessment Pipeline)
           → 📈 Dashboard (Visualization)
```

In [1]:
# ═══════════════════════════════════════════════════════════
# SECTION 1: Setup & Imports
# ═══════════════════════════════════════════════════════════
import sys
from pathlib import Path

# Add project root to path
project_root = str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Core imports
import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import date, datetime

print("✅ Core libraries imported successfully!")
print(f"📁 Project root: {project_root}")

✅ Core libraries imported successfully!
📁 Project root: c:\Users\AY11338\OneDrive - The Hartford\Desktop\Work DetailsARON\AI ML DS\MedGuard_AI


---
## 📄 Section 2: Generate Synthetic Medical Records (Data Layer)

We generate **50 synthetic patients** with realistic medical data including:
- Demographics, allergies, insurance
- Medications (including intentional dangerous drug combinations for GARMA to detect)
- Diagnoses with ICD-10 codes
- Vital signs history, lab results, encounter records

This uses FHIR-inspired Pydantic data models for validation.

In [2]:
# Reload modules after source code fix
import importlib
import medguard.data.generator
importlib.reload(medguard.data.generator)
from medguard.data.generator import generate_patient_dataset, save_dataset, load_dataset
print("✅ Modules reloaded")

✅ Modules reloaded


In [3]:
# ═══════════════════════════════════════════════════════════
# SECTION 2: Generate Synthetic Medical Records
# ═══════════════════════════════════════════════════════════
from medguard.data.generator import generate_patient_dataset, save_dataset, load_dataset
from medguard.data.models import PatientRecord

# Generate 50 patients (30% risky, 25% elderly)
patients = generate_patient_dataset(num_patients=50, risky_fraction=0.3, elderly_fraction=0.25)
save_dataset(patients)

# Quick stats
print(f"\n📊 Dataset Statistics:")
print(f"   Total patients: {len(patients)}")
print(f"   Average age: {sum(p.age for p in patients) / len(patients):.1f}")
print(f"   Male/Female: {sum(1 for p in patients if p.gender == 'Male')}/{sum(1 for p in patients if p.gender == 'Female')}")
print(f"   Patients on 4+ meds: {sum(1 for p in patients if len(p.active_medications) >= 4)}")
print(f"   With abnormal labs: {sum(1 for p in patients if any(lr.is_abnormal for lr in p.lab_results))}")

# Show a sample patient record
sample = patients[0]
print(f"\n{'='*60}")
print(f"📋 Sample Patient Record:")
print(f"{'='*60}")
print(sample.to_clinical_summary())

✅ Generated 50 patient records in c:\Users\AY11338\OneDrive - The Hartford\Desktop\Work DetailsARON\AI ML DS\MedGuard_AI\data\patients
📄 Clinical summaries saved in c:\Users\AY11338\OneDrive - The Hartford\Desktop\Work DetailsARON\AI ML DS\MedGuard_AI\data\summaries

📊 Dataset Statistics:
   Total patients: 50
   Average age: 61.3
   Male/Female: 27/23
   Patients on 4+ meds: 21
   With abnormal labs: 48

📋 Sample Patient Record:
PATIENT CLINICAL SUMMARY
Patient: Brian Yang (ID: 150ea6d6)
Age: 71 | Gender: Male | Blood Type: O+
PCP: Dr. Chapman, MD

ALLERGIES: Ibuprofen

ACTIVE MEDICATIONS:
  - Insulin Glargine 20 units (once daily at bedtime)
  - Amiodarone 200mg (once daily)
  - Simvastatin 40mg (once daily at bedtime)
  - Amiodarone 200mg (once daily)

DIAGNOSES:
  - Hypothyroidism (E03.9) — mild [CHRONIC]
  - Gastroesophageal Reflux Disease (K21.0) — mild [CHRONIC]
  - Type 2 Diabetes Mellitus (E11.9) — moderate [CHRONIC]
  - Chronic Low Back Pain (M54.5) — moderate [CHRONIC]

LATE

---
## 🤖 Section 3: JARVIS — AI-Augmented Engineering Agent

JARVIS enables **natural language querying** of the patient database. You can ask questions like:
- *"Find patients with diabetes"*
- *"Show me high-risk elderly patients"*  
- *"How many patients have abnormal labs?"*

It uses smart keyword matching, clinical category mapping, and relevance scoring — all **without needing an API key**.

In [4]:
# ═══════════════════════════════════════════════════════════
# SECTION 3: JARVIS — AI-Augmented Engineering Agent
# ═══════════════════════════════════════════════════════════
from medguard.jarvis.agent import JarvisAgent

jarvis = JarvisAgent()

# Demo queries
demo_queries = [
    "How many patients have diabetes?",
    "Find high-risk elderly patients",
    "Which patients are on Warfarin?",
    "Show me patients with chronic conditions",
    "Give me an overview of the database",
]

for query in demo_queries:
    print(f"\n{'🔍 ' + query}")
    print("─" * 60)
    response = jarvis.query(query)
    # Print first 20 lines to keep output manageable
    lines = response.split('\n')
    print('\n'.join(lines[:20]))
    if len(lines) > 20:
        print(f"  ... ({len(lines) - 20} more lines)")
    print()

🤖 JARVIS loaded 250 patient records

🔍 How many patients have diabetes?
────────────────────────────────────────────────────────────
🤖 JARVIS — Database Statistics
  Total patients: 250 (100%) ████████████████████
  Elderly (65+): 141 (56%) ███████████░░░░░░░░░
  With diabetes: 38 (15%) ███░░░░░░░░░░░░░░░░░
  With hypertension: 51 (20%) ████░░░░░░░░░░░░░░░░
  With abnormal labs: 242 (97%) ███████████████████░
  On 4+ medications: 107 (43%) ████████░░░░░░░░░░░░
  With chronic conditions: 233 (93%) ██████████████████░░


🔍 Find high-risk elderly patients
────────────────────────────────────────────────────────────
🤖 JARVIS — Found 10 matching patients for: 'Find high-risk elderly patients'

──────────────────────────────────────────────────
  #1 | Patricia Keller (ID: 0609cae3)
  Age: 83 | Gender: Female | Relevance: 8
  Match: Age: 83, Polypharmacy: 7 meds, Has abnormal labs
  Active Meds: Levothyroxine, Metoprolol, Insulin Glargine, Amiodarone
  Diagnoses: Gastroesophageal Reflux Disea

---
## 🛡️ Section 4: GARMA — GenAI Risk Mitigation Advisor

GARMA is the **risk analysis engine** that performs:
1. **Drug Interaction Detection** — Flags dangerous medication combinations (Warfarin+Aspirin, Opioids+Benzos, etc.)
2. **Compliance Checking** — Validates records against medical standards
3. **ML-Based Risk Scoring** — Weighted composite score across 6 clinical factors
4. **Anomaly Detection** — Polypharmacy, allergy conflicts, vital sign changes
5. **Recommendation Engine** — Actionable risk mitigation advice

In [5]:
# ═══════════════════════════════════════════════════════════
# SECTION 4: GARMA — Risk Assessment Pipeline
# ═══════════════════════════════════════════════════════════
from medguard.garma.advisor import GarmaAdvisor, RiskLevel

garma = GarmaAdvisor()

# Run GARMA on ALL patients
print("🛡️ Running GARMA Risk Assessment on all 50 patients...")
print("=" * 60)
assessments = garma.assess_all(patients)

# Population summary
summary = garma.get_population_risk_summary()
print(f"\n📊 GARMA Population Summary:")
print(f"   Total Assessed: {summary['total_assessed']}")
print(f"   Average Risk Score: {summary['avg_risk_score']:.1%}")
print(f"   Max Risk Score: {summary['max_risk_score']:.1%}")
print(f"   Drug Interactions Found: {summary['total_drug_interactions']}")
print(f"   Anomalies Detected: {summary['total_anomalies']}")
print(f"   Compliance Violations: {summary['total_compliance_violations']}")
print(f"\n   Risk Distribution:")
for level, count in summary['risk_distribution'].items():
    bar = "█" * count + "░" * (20 - count)
    print(f"     {level:10s}: {bar} {count}")

# Show the highest-risk patient's full report
print(f"\n{'='*60}")
print("🔴 HIGHEST RISK PATIENT — Full GARMA Report:")
print(f"{'='*60}")
highest = max(assessments, key=lambda a: a.overall_score)
print(garma.format_assessment_report(highest))

🛡️ Running GARMA Risk Assessment on all 50 patients...

📊 GARMA Population Summary:
   Total Assessed: 50
   Average Risk Score: 33.0%
   Max Risk Score: 72.1%
   Drug Interactions Found: 23
   Anomalies Detected: 194
   Compliance Violations: 0

   Risk Distribution:
     LOW       : ████████████████████████████████████████████████ 48
     MEDIUM    : ██░░░░░░░░░░░░░░░░░░ 2
     HIGH      : ░░░░░░░░░░░░░░░░░░░░ 0
     CRITICAL  : ░░░░░░░░░░░░░░░░░░░░ 0

🔴 HIGHEST RISK PATIENT — Full GARMA Report:
════════════════════════════════════════════════════════════
🛡️  GARMA RISK ASSESSMENT REPORT
════════════════════════════════════════════════════════════
Patient: Lauren Armstrong (ID: d634fc05)

OVERALL RISK: 🟡 MEDIUM
Risk Score: 72.1%

────────────────────────────────────────
📊 RISK FACTOR BREAKDOWN:
  age_over_65               █████████████░░░░░░░ 68%
  multiple_conditions       ████████████████████ 100%
  drug_interactions         ████████████████░░░░ 80%
  recent_hospitalization    ░░░░

---
## 📈 Section 5: Interactive Visualizations

Rich Plotly visualizations showing risk distributions, age patterns, and population analytics.

In [6]:
# ═══════════════════════════════════════════════════════════
# SECTION 5: Visualizations
# ═══════════════════════════════════════════════════════════

# Build a comprehensive DataFrame for visualization
viz_data = []
for a in assessments:
    p = next((p for p in patients if p.patient_id == a.patient_id), None)
    if p:
        viz_data.append({
            "Patient": p.full_name,
            "Age": p.age,
            "Gender": p.gender,
            "Risk Score": a.overall_score,
            "Risk Level": a.risk_level.value,
            "Active Meds": len(p.active_medications),
            "Diagnoses": len(p.diagnoses),
            "Drug Interactions": len(a.drug_interactions),
            "Anomalies": len(a.anomalies),
            "Chronic Conditions": len(p.chronic_conditions),
        })

df = pd.DataFrame(viz_data)

# ── Chart 1: Risk Distribution Pie Chart ──
fig1 = px.pie(
    df, names="Risk Level",
    title="🛡️ GARMA Risk Level Distribution",
    color="Risk Level",
    color_discrete_map={"LOW": "#4caf50", "MEDIUM": "#ff9800", "HIGH": "#f44336", "CRITICAL": "#9c27b0"},
    hole=0.4,
)
fig1.update_layout(height=400)
fig1.show()

# ── Chart 2: Risk Score vs Age Scatter ──
fig2 = px.scatter(
    df, x="Age", y="Risk Score",
    color="Risk Level", size="Active Meds",
    hover_data=["Patient", "Drug Interactions"],
    title="📊 Patient Risk Score vs Age (bubble size = # medications)",
    color_discrete_map={"LOW": "#4caf50", "MEDIUM": "#ff9800", "HIGH": "#f44336", "CRITICAL": "#9c27b0"},
)
fig2.add_hline(y=0.6, line_dash="dash", line_color="orange", annotation_text="Medium Risk Threshold")
fig2.add_hline(y=0.8, line_dash="dash", line_color="red", annotation_text="High Risk Threshold")
fig2.update_layout(height=500)
fig2.show()

# ── Chart 3: Risk Score Histogram ──
fig3 = px.histogram(
    df, x="Risk Score", nbins=20, color="Risk Level",
    title="📈 Risk Score Distribution",
    color_discrete_map={"LOW": "#4caf50", "MEDIUM": "#ff9800", "HIGH": "#f44336", "CRITICAL": "#9c27b0"},
)
fig3.update_layout(height=400)
fig3.show()

# ── Chart 4: Most Common Conditions ──
condition_counts = {}
for p in patients:
    for d in p.diagnoses:
        condition_counts[d.condition] = condition_counts.get(d.condition, 0) + 1

cond_df = pd.DataFrame([
    {"Condition": k, "Count": v}
    for k, v in sorted(condition_counts.items(), key=lambda x: x[1], reverse=True)[:10]
])

fig4 = px.bar(
    cond_df, x="Count", y="Condition", orientation="h",
    title="🏥 Top 10 Most Common Conditions",
    color="Count", color_continuous_scale="blues",
)
fig4.update_layout(height=450, yaxis=dict(autorange="reversed"))
fig4.show()

print(f"\n✅ All visualizations rendered! ({len(df)} patients analyzed)")


✅ All visualizations rendered! (50 patients analyzed)


---
## 📊 Section 6: NotebookLM-Style Semantic Search

The NotebookLM module ingests all clinical summaries, chunks them by medical section, creates vector embeddings using **sentence-transformers** (free, local), stores them in **FAISS**, and enables semantic search.

> **Note:** This section requires `sentence-transformers` and `faiss-cpu`. If not installed, it will fall back to TF-IDF based search.

In [7]:
# ═══════════════════════════════════════════════════════════
# SECTION 6: NotebookLM — Semantic Search Demo
# ═══════════════════════════════════════════════════════════
try:
    from medguard.notebooklm.engine import NotebookLMEngine

    nlm = NotebookLMEngine()
    nlm.ingest_documents()

    # Semantic search queries
    search_queries = [
        "patients with diabetes and abnormal glucose",
        "high blood pressure medication interactions",
        "elderly patient on warfarin with bleeding risk",
        "depression treatment with SSRI medications",
    ]

    for query in search_queries:
        print(f"\n🔍 Query: '{query}'")
        print("─" * 50)
        results = nlm.search(query, top_k=3)
        for i, r in enumerate(results, 1):
            score = r.get("similarity_score", 0)
            doc_id = r.get("doc_id", "?")
            section = r.get("section", "?")
            text_preview = r.get("text", "")[:150]
            print(f"  [{i}] Score: {score:.3f} | Doc: {doc_id} | Section: {section}")
            print(f"      {text_preview}...")
        print()

    # Corpus insights
    insights = nlm.get_document_insights()
    print(f"\n📊 Knowledge Base Insights:")
    for k, v in insights.items():
        print(f"   {k}: {v}")

except Exception as e:
    print(f"⚠️ NotebookLM demo skipped (install sentence-transformers & faiss-cpu): {e}")
    print("   Run: pip install sentence-transformers faiss-cpu")

📄 Ingesting 152 documents...
📦 Created 1008 chunks from 152 documents
🔨 Building vector index for 1008 chunks...
⚠️ sentence-transformers not installed. Using TF-IDF fallback.
🔨 Building vector index for 1008 chunks...
⚠️ sentence-transformers not installed. Using TF-IDF fallback.
✅ Vector index built: 1008 vectors, 384D
💾 Vector store saved to c:\Users\AY11338\OneDrive - The Hartford\Desktop\Work DetailsARON\AI ML DS\MedGuard_AI\data\vector_store

🔍 Query: 'patients with diabetes and abnormal glucose'
──────────────────────────────────────────────────
  [1] Score: 0.266 | Doc: e0d88787_summary | Section: DIAGNOSES
      :
  - Type 2 Diabetes Mellitus (E11.9) — moderate [CHRONIC]
  - Urinary Tract Infection (N39.0) — mild...
  [2] Score: 0.266 | Doc: 97a9426a_summary | Section: DIAGNOSES
      :
  - Type 2 Diabetes Mellitus (E11.9) — moderate [CHRONIC]
  - Urinary Tract Infection (N39.0) — mild...
  [3] Score: 0.266 | Doc: 6a303b20_summary | Section: DIAGNOSES
      :
  - Type 2 Diabet

---
## ✅ Section 7: End-to-End Pipeline — Full Walkthrough

Combining all modules: **JARVIS** queries → **GARMA** risk assessment → **NotebookLM** knowledge grounding → **Visualization**

In [8]:
# ═══════════════════════════════════════════════════════════
# SECTION 7: End-to-End Pipeline Walkthrough
# ═══════════════════════════════════════════════════════════

print("🚀 MedGuard AI — Full Pipeline Demo")
print("=" * 60)

# Step 1: JARVIS finds relevant patients
print("\n🤖 Step 1: JARVIS — Finding patients with diabetes on multiple medications...")
results = jarvis.search_patients("diabetes patients on multiple medications", max_results=3)

for r in results:
    p = r["patient"]
    print(f"\n  📋 {p.full_name} (ID: {p.patient_id})")
    
    # Step 2: GARMA assesses each found patient
    assessment = garma.assessments.get(p.patient_id)
    if assessment:
        risk_emoji = {"LOW": "🟢", "MEDIUM": "🟡", "HIGH": "🟠", "CRITICAL": "🔴"}
        print(f"  🛡️ GARMA Risk: {risk_emoji.get(assessment.risk_level.value, '⚪')} {assessment.risk_level.value} ({assessment.overall_score:.1%})")
        
        if assessment.drug_interactions:
            print(f"  💊 Drug Interactions: {len(assessment.drug_interactions)}")
            for di in assessment.drug_interactions:
                print(f"     ⚠️ {di.drug_a} + {di.drug_b} [{di.severity}]")
        
        if assessment.anomalies:
            print(f"  🔍 Anomalies: {len(assessment.anomalies)}")
            for anom in assessment.anomalies[:2]:
                print(f"     {anom}")
        
        if assessment.recommendations:
            print(f"  ✅ Top Recommendation: {assessment.recommendations[0]}")

# Summary visualization
print(f"\n{'='*60}")
print("📈 Pipeline Summary — Risk Assessment Radar Chart")

# Create radar chart for the highest-risk patient
highest = max(assessments, key=lambda a: a.overall_score)
categories = list(highest.risk_factors.keys())
values = list(highest.risk_factors.values())

fig = go.Figure(data=go.Scatterpolar(
    r=values + [values[0]],
    theta=[c.replace("_", " ").title() for c in categories] + [categories[0].replace("_", " ").title()],
    fill='toself',
    name=f"Risk Profile: {highest.patient_name}",
    line_color='red',
    fillcolor='rgba(255, 0, 0, 0.2)',
))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title=f"🛡️ GARMA Risk Radar — {highest.patient_name} (Score: {highest.overall_score:.1%})",
    height=500,
)
fig.show()

print("\n✅ MedGuard AI Pipeline Complete!")
print("🚀 Launch the full dashboard: streamlit run medguard/dashboard/app.py")

🚀 MedGuard AI — Full Pipeline Demo

🤖 Step 1: JARVIS — Finding patients with diabetes on multiple medications...

  📋 Omar Alvarado (ID: 34bc93a8)

  📋 Omar Alvarado (ID: 858156f1)

  📋 Omar Alvarado (ID: ab8a2f05)

📈 Pipeline Summary — Risk Assessment Radar Chart



✅ MedGuard AI Pipeline Complete!
🚀 Launch the full dashboard: streamlit run medguard/dashboard/app.py
